In [ ]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.2/187.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.2/93.2 kB 9.8 MB/s eta 0:00:00


In [ ]:
import gymnasium as gym
import numpy as np
import pandas as pd
from gymnasium import spaces
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
import random

# ==========================================
# 1. CONFIGURATION
# ==========================================

CONFIG = {
    "days": 14,
    "shifts_per_day": 5,
    "staff_per_shift": 2,
    "min_rest_hours": 12,
    "currency_threshold": 50,
    "n_controllers": 20,
    "shift_duration": 4.8,
    "max_consecutive_days": 4,
    "absenteeism_rate": 0.30
}

class Controller:
    def __init__(self, uid, qualifications):
        self.uid = uid
        self.qualifications = qualifications
        self.history = []
        self.last_shift_end_time = -999.0
        self.days_since_last_role = {q: 0 for q in qualifications}

        self.unavailable_days = set()
        self.consecutive_days_worked = 0
        self.last_day_worked = -99

    def reset(self):
        self.history = []
        self.last_shift_end_time = -999.0
        self.days_since_last_role = {q: random.randint(0, 30) for q in self.qualifications}
        self.unavailable_days = set()
        self.consecutive_days_worked = 0
        self.last_day_worked = -99

    def increment_currency(self):
        for q in self.days_since_last_role:
            self.days_since_last_role[q] += 1

# ==========================================
# 2. ENVIRONMENT
# ==========================================

class ATCRosteringEnv(gym.Env):
    def __init__(self):
        super(ATCRosteringEnv, self).__init__()
        self.n_controllers = CONFIG["n_controllers"]
        self.total_slots = CONFIG["days"] * CONFIG["shifts_per_day"] * CONFIG["staff_per_shift"]
        self.action_space = spaces.Discrete(self.n_controllers + 1)
        self.obs_dim = 3 + (self.n_controllers * 5)
        self.observation_space = spaces.Box(low=0, high=1, shape=(self.obs_dim,), dtype=np.float32)
        self.controllers = []
        self._generate_controllers()
        self.current_step = 0
        self.roster = []

    def _generate_controllers(self):
        self.controllers = []
        all_shifts = range(CONFIG["shifts_per_day"])
        specialists = []
        for s in all_shifts:
            for _ in range(4): specialists.append(s)

        for i in range(self.n_controllers):
            quals = set()
            if i < len(specialists): quals.add(specialists[i])
            num_extra = random.randint(2, 5)
            quals.update(random.sample(all_shifts, min(len(all_shifts), num_extra)))
            self.controllers.append(Controller(i, list(quals)))

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        self.roster = []
        for c in self.controllers:
            c.reset()
            if random.random() < CONFIG["absenteeism_rate"]:
                num_sick_days = random.randint(1, 3)
                start_day = random.randint(0, CONFIG["days"] - 2)
                sick_days = set(range(start_day, min(CONFIG["days"], start_day + num_sick_days)))
                c.unavailable_days.update(sick_days)

        # Ensure at least one person is sick on Day 2 for testing
        if not any(2 in c.unavailable_days for c in self.controllers):
             self.controllers[0].unavailable_days.add(2)
        return self._get_observation(), {}

    def _get_context(self):
        slots_per_shift = CONFIG["staff_per_shift"]
        slots_per_day = CONFIG["shifts_per_day"] * slots_per_shift
        day = self.current_step // slots_per_day
        remainder = self.current_step % slots_per_day
        shift_idx = remainder // slots_per_shift
        slot_idx = remainder % slots_per_shift
        current_time_abs = (day * 24) + (shift_idx * CONFIG["shift_duration"])
        return day, shift_idx, slot_idx, current_time_abs

    def _get_observation(self):
        day, shift_idx, slot_idx, current_time_abs = self._get_context()
        obs = [day / CONFIG["days"], shift_idx / CONFIG["shifts_per_day"], slot_idx / CONFIG["staff_per_shift"]]
        for c in self.controllers:
            if c.last_shift_end_time < 0: hours_rest = 48.0
            else: hours_rest = current_time_abs - c.last_shift_end_time
            norm_rest = min(hours_rest, 48.0) / 48.0
            is_qual = 1.0 if shift_idx in c.qualifications else 0.0
            days_since = c.days_since_last_role.get(shift_idx, 0)
            norm_curr = min(days_since, CONFIG["currency_threshold"]) / CONFIG["currency_threshold"]
            is_absent = 1.0 if day in c.unavailable_days else 0.0
            norm_cons = c.consecutive_days_worked / CONFIG["max_consecutive_days"]
            obs.extend([norm_rest, is_qual, norm_curr, is_absent, norm_cons])
        return np.array(obs, dtype=np.float32)

    def valid_action_mask(self):
        day, shift_idx, _, current_time_abs = self._get_context()
        mask = np.zeros(self.action_space.n, dtype=bool)
        mask[-1] = True
        for i, c in enumerate(self.controllers):
            if shift_idx not in c.qualifications:
                mask[i] = False; continue
            if day in c.unavailable_days:
                mask[i] = False; continue
            if c.consecutive_days_worked >= CONFIG["max_consecutive_days"]:
                 if c.last_day_worked == day - 1:
                     mask[i] = False; continue
            if c.days_since_last_role.get(shift_idx, 0) >= CONFIG["currency_threshold"]:
                mask[i] = False; continue
            if c.last_shift_end_time >= 0:
                gap = current_time_abs - c.last_shift_end_time
                if gap < CONFIG["min_rest_hours"]:
                    mask[i] = False; continue
            already_working = False
            for r_day, r_shift, r_id in self.roster:
                if r_day == day and r_shift == shift_idx and r_id == i:
                    already_working = True; break
            if already_working: mask[i] = False; continue
            mask[i] = True
        return mask

    def step(self, action):
        day, shift_idx, slot_idx, current_time_abs = self._get_context()
        reward = 0
        if action == self.n_controllers:
            reward -= 100
            self.roster.append((day, shift_idx, None))
        else:
            c = self.controllers[action]
            reward += 10
            if shift_idx == 4:
                prev_night = False
                for h_day, h_shift in c.history:
                    if h_day == (day - 1) and h_shift == 4:
                        prev_night = True; break
                if prev_night: reward -= 5

            if c.last_day_worked == day - 1: c.consecutive_days_worked += 1
            elif c.last_day_worked < day - 1: c.consecutive_days_worked = 1
            elif c.last_day_worked != day: c.consecutive_days_worked = 1

            c.last_day_worked = day
            c.history.append((day, shift_idx))
            c.last_shift_end_time = current_time_abs + CONFIG["shift_duration"]
            c.days_since_last_role[shift_idx] = 0
            self.roster.append((day, shift_idx, c.uid))

        self.current_step += 1
        if self.current_step % (CONFIG["shifts_per_day"] * CONFIG["staff_per_shift"]) == 0:
            for c in self.controllers: c.increment_currency()

        done = self.current_step >= self.total_slots
        return self._get_observation(), reward, done, False, {}

# ==========================================
# 3. COMPREHENSIVE REPORTING SUITE
# ==========================================

def print_workforce_metadata(env):
    print("\n" + "="*60)
    print("                 WORKFORCE AVAILABILITY REPORT")
    print("="*60)
    print(f"{'ID':<4} | {'Qualifications (Shifts)':<25} | {'Unavailable (Sick Days)'}")
    print("-" * 60)
    sick_count = 0
    for c in env.unwrapped.controllers:
        sick = str(sorted(list(c.unavailable_days))) if c.unavailable_days else "Available"
        if c.unavailable_days: sick_count += 1
        quals = str(sorted(c.qualifications))
        print(f"C_{c.uid:<2} | {quals:<25} | {sick}")
    print("="*60 + "\n")

def validate_full_schedule(df, env):
    print("\n" + "="*60)
    print("               FULL SCHEDULE HEALTH CHECK")
    print("="*60)

    violations = { "Fatigue": 0, "Qualifications": 0, "Absenteeism": 0, "Overwork": 0, "DoubleBooking": 0 }
    controllers = {f"C_{c.uid}": c for c in env.unwrapped.controllers}

    # Check Coverage
    empty = len(df[df["Controller"] == "EMPTY"])
    total = len(df)
    print(f"Coverage: {(total-empty)/total*100:.1f}% ({total-empty}/{total} slots filled)")

    assignments = df[df["Controller"] != "EMPTY"].copy()
    assignments["Day"] = assignments["Day"].astype(int)
    assignments["Shift"] = assignments["Shift"].astype(int)

    for c_name, c_obj in controllers.items():
        c_shifts = assignments[assignments["Controller"] == c_name].sort_values(["Day", "Shift"])
        if c_shifts.empty: continue

        prev_end = -999.0

        # Check each shift
        for _, row in c_shifts.iterrows():
            d, s = row["Day"], row["Shift"]
            if s not in c_obj.qualifications: violations["Qualifications"] += 1
            if d in c_obj.unavailable_days: violations["Absenteeism"] += 1

            start = (d * 24) + (s * CONFIG["shift_duration"])
            if prev_end != -999.0:
                gap = start - prev_end
                if gap < CONFIG["min_rest_hours"] - 0.01:
                    if gap < 0.1: violations["DoubleBooking"] += 1
                    else: violations["Fatigue"] += 1
            prev_end = start + CONFIG["shift_duration"]

        # Check streaks
        days = sorted(c_shifts["Day"].unique())
        cons = 0
        prev_d = -99
        for day in days:
            if day == prev_d + 1: cons += 1
            else: cons = 1
            if cons > CONFIG["max_consecutive_days"]: violations["Overwork"] += 1
            prev_d = day

    all_pass = sum(violations.values()) == 0
    for k, v in violations.items():
        status = "PASS" if v == 0 else "FAIL"
        print(f"{k:<20}: {v} [{status}]")

    print("="*60)
    if all_pass and empty == 0: print("RESULT: ROSTER IS PERFECT.")
    else: print("RESULT: WARNINGS FOUND.")

def analyze_edge_cases(df, env):
    print("\n" + "="*80)
    print("                   DEEP DIVE: EDGE CASE SCENARIOS")
    print("="*80)

    controllers = {f"C_{c.uid}": c for c in env.unwrapped.controllers}
    df["Day"] = df["Day"].astype(int)

    # --- CASE 1: ABSENTEEISM ---
    print("SCENARIO 1: The 'Sick Day' Check")
    print("(Verifies user is NOT scheduled on unavailable days)")
    sick_staff = [c for c in env.unwrapped.controllers if c.unavailable_days]
    target = sick_staff[0] if sick_staff else None

    if target:
        sick_days = sorted(list(target.unavailable_days))
        print(f" -> Subject: Controller C_{target.uid}")
        print(f" -> Sick Days: {sick_days}")
        assigned = df[df["Controller"] == f"C_{target.uid}"]["Day"].unique()
        fails = [d for d in assigned if d in sick_days]

        timeline = ""
        for d in range(CONFIG["days"]):
            if d in sick_days: m = "[SICK]"
            elif d in assigned: m = "[Work]"
            else: m = " .... "
            timeline += f"D{d}:{m} "
        print(f" -> Timeline: {timeline}")
        print(f" -> RESULT: {'FAIL' if fails else 'PASS'}")
    else:
        print(" -> No sick staff found.")
    print("-" * 80)

    # --- CASE 2: MAX CONSECUTIVE DAYS ---
    print("SCENARIO 2: The 'Burnout' Check")
    print(f"(Verifies user stops working after {CONFIG['max_consecutive_days']} consecutive days)")
    candidates = []
    for uid, c in controllers.items():
        c_shifts = df[df["Controller"] == uid].sort_values("Day")
        if c_shifts.empty: continue
        days = sorted(c_shifts["Day"].unique())
        max_streak, curr, prev = 0, 0, -99
        for d in days:
            if d == prev + 1: curr += 1
            else: curr = 1
            max_streak = max(max_streak, curr)
            prev = d
        candidates.append((uid, max_streak, days))

    candidates.sort(key=lambda x: x[1], reverse=True)

    if candidates:
        uid, streak, days = candidates[0]
        print(f" -> Subject: {uid} (Max Streak Found: {streak})")
        timeline = ""
        for d in range(CONFIG["days"]):
            m = "[Work]" if d in days else "[REST]"
            timeline += f"D{d}:{m} "
        print(f" -> Timeline: {timeline}")
        print(f" -> RESULT: {'PASS' if streak <= CONFIG['max_consecutive_days'] else 'FAIL'}")
    print("-" * 80)

    # --- CASE 3: FATIGUE (NIGHT SHIFT RECOVERY) ---
    print("SCENARIO 3: The 'Clopen' Check (Fatigue)")
    print("(Verifies NO Morning Shift (0 or 1) after a Night Shift (4))")

    night_shifters = df[df["Shift"] == 4]["Controller"].unique()
    night_shifters = [n for n in night_shifters if n != "EMPTY"]

    target_ns = night_shifters[0] if len(night_shifters) > 0 else None

    if target_ns:
        print(f" -> Subject: {target_ns}")
        c_sched = df[df["Controller"] == target_ns].sort_values(["Day", "Shift"])

        # Find a day they worked shift 4
        ns_days = c_sched[c_sched["Shift"] == 4]["Day"].values
        example_day = ns_days[0]

        print(f" -> Worked Night Shift (4) on Day {example_day}")
        # Check next day
        next_day_sched = c_sched[c_sched["Day"] == example_day + 1]
        morning_work = next_day_sched[next_day_sched["Shift"] < 2]

        if morning_work.empty:
             print(f" -> Day {example_day+1} Schedule: {next_day_sched['Shift'].values if not next_day_sched.empty else 'REST'}")
             print(" -> RESULT: PASS (Rest period respected)")
        else:
             print(f" -> Day {example_day+1} Schedule: {morning_work['Shift'].values} (VIOLATION!)")
             print(" -> RESULT: FAIL")
    else:
        print(" -> No night shifts assigned.")

    print("-" * 80)

    # --- CASE 4: QUALIFICATIONS ---
    print("SCENARIO 4: The 'Specialist' Check")
    print("(Verifies user has the specific license for the assigned shift)")

    # Pick a random non-empty assignment
    assigned_slots = df[df["Controller"] != "EMPTY"]
    if not assigned_slots.empty:
        sample = assigned_slots.sample(1).iloc[0]
        uid = sample["Controller"]
        shift = sample["Shift"]
        day = sample["Day"]

        c_obj = controllers[uid]
        print(f" -> Checking Assignment: {uid} on Day {day}, Shift {shift}")
        print(f" -> Controller Qualifications: {sorted(c_obj.qualifications)}")

        if shift in c_obj.qualifications:
            print(" -> RESULT: PASS (Qualified)")
        else:
            print(" -> RESULT: FAIL (Not Qualified)")
    else:
        print(" -> No assignments made.")

    print("="*80)

# ==========================================
# 4. MAIN
# ==========================================

def mask_fn(env: gym.Env) -> np.ndarray: return env.valid_action_mask()

In [ ]:
def main():
    env = ATCRosteringEnv()
    env = ActionMasker(env, mask_fn)
    model = MaskablePPO("MlpPolicy", env, gamma=0.95, learning_rate=1e-3, n_steps=2048, batch_size=128, ent_coef=0.01, verbose=0)

    print(f"Training Model on {CONFIG['days']} day schedule...")
    model.learn(total_timesteps=60000)

    print("\nGenerating Roster...")
    obs, _ = env.reset()
    done = False
    while not done:
        action_masks = env.action_masks()
        action, _ = model.predict(obs, action_masks=action_masks, deterministic=True)
        obs, reward, done, _, _ = env.step(action)

    roster_raw = env.unwrapped.roster
    clean = [{"Day": r[0], "Shift": r[1], "Controller": f"C_{r[2]}" if r[2] is not None else "EMPTY"} for r in roster_raw]
    df = pd.DataFrame(clean)
    pivot = df.groupby(['Day', 'Shift'])['Controller'].apply(list).unstack()

    # 1. METADATA
    print_workforce_metadata(env)

    # 2. ROSTER
    print("\n=== GENERATED ROSTER ===")
    print(pivot.to_string())

    # 3. HEALTH CHECK
    validate_full_schedule(df, env)

    # 4. CASE STUDIES
    analyze_edge_cases(df, env)

if __name__ == "__main__":
    main()

Training Model on 10 day schedule...

Generating Roster...

                 WORKFORCE AVAILABILITY REPORT
ID   | Qualifications (Shifts)   | Unavailable (Sick Days)
------------------------------------------------------------
C_0  | [0, 1, 4]                 | [0]
C_1  | [0, 1, 2, 3]              | Available
C_2  | [0, 2, 4]                 | [2, 3, 4]
C_3  | [0, 1, 2, 3, 4]           | [1]
C_4  | [0, 1, 2, 3, 4]           | Available
C_5  | [0, 1, 2, 3, 4]           | Available
C_6  | [1, 3, 4]                 | [2, 3, 4]
C_7  | [1, 2, 4]                 | [3, 4, 5]
C_8  | [0, 2, 3]                 | Available
C_9  | [1, 2, 3]                 | Available
C_10 | [0, 1, 2]                 | Available
C_11 | [2, 3, 4]                 | Available
C_12 | [0, 2, 3]                 | [0, 1, 2]
C_13 | [0, 2, 3]                 | Available
C_14 | [0, 2, 3]                 | Available
C_15 | [0, 1, 3]                 | Available
C_16 | [0, 3, 4]                 | Available
C_17 | [0, 2, 3, 4] 

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
